# FreshGuard v2.1 - 00 Fetch Official Sources

Run this Kaggle notebook first with **Internet On** and **no accelerator**. It creates the reusable official-source dataset that downstream v2/v2.1 notebooks attach as `freshguard-official-sources-v2`.

Official sources:
- KTH GroceryStoreDataset: https://github.com/marcusklasson/GroceryStoreDataset
- Open Images V7: https://storage.googleapis.com/openimages/web/download_v7.html
- FiftyOne Open Images loader: https://docs.voxel51.com/dataset_zoo/datasets/open_images_v7.html

v2.1 adds Open Images negative/background samples so the detector learns that arbitrary non-produce images should produce empty YOLO labels instead of full-frame `produce` boxes.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

OUT_DIR = Path('/kaggle/working/freshguard_official_sources_v2')
GROCERY_OUT = OUT_DIR / 'grocery_store_dataset'
OPEN_IMAGES_OUT = OUT_DIR / 'open_images_produce'
OPEN_IMAGES_IMAGE_DIR = OPEN_IMAGES_OUT / 'images'
for split in ('train', 'val', 'test'):
    (OPEN_IMAGES_IMAGE_DIR / split).mkdir(parents=True, exist_ok=True)
    (OPEN_IMAGES_IMAGE_DIR / 'negative' / split).mkdir(parents=True, exist_ok=True)

OPEN_IMAGES_CLASSES = ['Apple', 'Banana', 'Orange', 'Tomato', 'Potato', 'Carrot', 'Cucumber', 'Strawberry', 'Mango', 'Bell pepper']
OPEN_IMAGES_NEGATIVE_ANCHORS = ['Car', 'Person', 'Chair', 'Table', 'Building', 'Computer monitor', 'Laptop', 'Book', 'Clock', 'Television', 'Bus', 'Train', 'Dog', 'Cat', 'Bicycle', 'Boat']
OPEN_IMAGES_SPLITS = {'train': 'train', 'validation': 'val', 'test': 'test'}
OPEN_IMAGES_MAX_SAMPLES = int(os.environ.get('OPEN_IMAGES_MAX_SAMPLES_PER_SPLIT', '2500'))
OPEN_IMAGES_NEGATIVE_MAX_SAMPLES = int(os.environ.get('OPEN_IMAGES_NEGATIVE_MAX_SAMPLES_PER_SPLIT', '1200'))

OUT_DIR.mkdir(parents=True, exist_ok=True)
print({'out_dir': str(OUT_DIR), 'open_images_max_samples_per_split': OPEN_IMAGES_MAX_SAMPLES, 'open_images_negative_max_samples_per_split': OPEN_IMAGES_NEGATIVE_MAX_SAMPLES})


In [ ]:
tmp_grocery = Path('/kaggle/working/GroceryStoreDataset')
if tmp_grocery.exists():
    shutil.rmtree(tmp_grocery)
subprocess.run(
    ['git', 'clone', '--depth', '1', 'https://github.com/marcusklasson/GroceryStoreDataset.git', str(tmp_grocery)],
    check=True,
)
if not (tmp_grocery / 'dataset').exists():
    raise RuntimeError('KTH GroceryStoreDataset clone is missing the dataset/ directory.')
if GROCERY_OUT.exists():
    shutil.rmtree(GROCERY_OUT)
shutil.copytree(tmp_grocery, GROCERY_OUT)
print({'grocery_root': str(GROCERY_OUT), 'files': len(list(GROCERY_OUT.rglob('*')))})

In [ ]:
try:
    import fiftyone as fo
    import fiftyone.zoo as foz
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fiftyone'], check=True)
    import fiftyone as fo
    import fiftyone.zoo as foz

import pandas as pd

fo.config.show_progress_bars = True


def classification_labels(sample, field_name: str) -> set[str]:
    labels = getattr(sample, field_name, None)
    if labels is None:
        return set()
    return {item.label for item in getattr(labels, 'classifications', [])}


def detection_labels(sample) -> set[str]:
    labels = getattr(sample, 'ground_truth', None)
    if labels is None:
        return set()
    return {item.label for item in getattr(labels, 'detections', [])}


def copy_sample_image(sample, output_split: str, *, negative: bool = False) -> Path:
    source = Path(sample.filepath)
    target_dir = OPEN_IMAGES_IMAGE_DIR / ('negative' if negative else '') / output_split
    target_dir.mkdir(parents=True, exist_ok=True)
    dest = target_dir / source.name
    if not dest.exists():
        shutil.copy2(source, dest)
    return dest

boxes: list[dict[str, object]] = []
for source_split, output_split in OPEN_IMAGES_SPLITS.items():
    dataset_name = f'freshguard-open-images-v7-produce-{source_split}'
    if fo.dataset_exists(dataset_name):
        fo.delete_dataset(dataset_name)
    dataset = foz.load_zoo_dataset(
        'open-images-v7',
        split=source_split,
        label_types=['detections'],
        classes=OPEN_IMAGES_CLASSES,
        max_samples=OPEN_IMAGES_MAX_SAMPLES,
        shuffle=True,
        dataset_name=dataset_name,
    )
    for sample in dataset:
        dest = copy_sample_image(sample, output_split)
        detections = getattr(sample, 'ground_truth', None)
        if detections is None:
            continue
        for detection in detections.detections:
            if detection.label not in OPEN_IMAGES_CLASSES:
                continue
            x, y, w, h = detection.bounding_box
            boxes.append(
                {
                    'path': str(dest),
                    'image_id': Path(sample.filepath).stem,
                    'split': output_split,
                    'open_images_label': detection.label,
                    'x1': float(x),
                    'y1': float(y),
                    'x2': float(x + w),
                    'y2': float(y + h),
                    'coordinate_format': 'normalized_xyxy',
                }
            )

if not boxes:
    raise RuntimeError('Open Images V7 download produced zero usable produce boxes.')

boxes_df = pd.DataFrame(boxes)
boxes_df.to_csv(OPEN_IMAGES_OUT / 'boxes.csv', index=False)
print({'open_images_boxes': len(boxes_df), 'by_split': boxes_df['split'].value_counts().to_dict()})

negative_rows: list[dict[str, object]] = []
negative_sources: dict[str, int] = {'verified_produce_negative': 0, 'non_produce_anchor_filtered': 0}


def collect_negative_rows(dataset, output_split: str, source_name: str, require_verified_negative: bool) -> None:
    for sample in dataset:
        positive_produce = classification_labels(sample, 'positive_labels') & set(OPEN_IMAGES_CLASSES)
        negative_produce = classification_labels(sample, 'negative_labels') & set(OPEN_IMAGES_CLASSES)
        produce_detections = detection_labels(sample) & set(OPEN_IMAGES_CLASSES)
        if positive_produce or produce_detections:
            continue
        if require_verified_negative and not negative_produce:
            continue
        dest = copy_sample_image(sample, output_split, negative=True)
        negative_rows.append(
            {
                'path': str(dest),
                'image_id': Path(sample.filepath).stem,
                'split': output_split,
                'negative_source': source_name,
                'verified_absent_produce_labels': '|'.join(sorted(negative_produce)),
                'anchor_labels': '|'.join(sorted((classification_labels(sample, 'positive_labels') | detection_labels(sample)) - set(OPEN_IMAGES_CLASSES))),
            }
        )
        negative_sources[source_name] += 1

for source_split, output_split in OPEN_IMAGES_SPLITS.items():
    dataset_name = f'freshguard-open-images-v7-produce-negative-{source_split}'
    if fo.dataset_exists(dataset_name):
        fo.delete_dataset(dataset_name)
    dataset = foz.load_zoo_dataset(
        'open-images-v7',
        split=source_split,
        label_types=['classifications', 'detections'],
        classes=OPEN_IMAGES_CLASSES,
        max_samples=OPEN_IMAGES_NEGATIVE_MAX_SAMPLES,
        shuffle=True,
        dataset_name=dataset_name,
    )
    collect_negative_rows(dataset, output_split, 'verified_produce_negative', require_verified_negative=True)

if not negative_rows:
    for source_split, output_split in OPEN_IMAGES_SPLITS.items():
        dataset_name = f'freshguard-open-images-v7-anchor-negative-{source_split}'
        if fo.dataset_exists(dataset_name):
            fo.delete_dataset(dataset_name)
        dataset = foz.load_zoo_dataset(
            'open-images-v7',
            split=source_split,
            label_types=['classifications', 'detections'],
            classes=OPEN_IMAGES_NEGATIVE_ANCHORS,
            max_samples=OPEN_IMAGES_NEGATIVE_MAX_SAMPLES,
            shuffle=True,
            dataset_name=dataset_name,
        )
        collect_negative_rows(dataset, output_split, 'non_produce_anchor_filtered', require_verified_negative=False)

if not negative_rows:
    raise RuntimeError('Open Images V7 download produced zero usable negative/background images.')

negatives_df = pd.DataFrame(negative_rows).drop_duplicates(subset=['path'])
negatives_df.to_csv(OPEN_IMAGES_OUT / 'negatives.csv', index=False)
print({'open_images_negatives': len(negatives_df), 'by_split': negatives_df['split'].value_counts().to_dict(), 'negative_sources': negative_sources})


In [ ]:
summary = {
    'grocery_source': 'https://github.com/marcusklasson/GroceryStoreDataset',
    'open_images_source': 'open-images-v7 via FiftyOne',
    'open_images_classes': OPEN_IMAGES_CLASSES,
    'open_images_negative_anchors': OPEN_IMAGES_NEGATIVE_ANCHORS,
    'open_images_max_samples_per_split': OPEN_IMAGES_MAX_SAMPLES,
    'open_images_negative_max_samples_per_split': OPEN_IMAGES_NEGATIVE_MAX_SAMPLES,
    'grocery_root': str(GROCERY_OUT),
    'open_images_boxes_csv': str(OPEN_IMAGES_OUT / 'boxes.csv'),
    'open_images_negatives_csv': str(OPEN_IMAGES_OUT / 'negatives.csv'),
    'open_images_box_count': int(len(boxes_df)),
    'open_images_negative_count': int(len(negatives_df)),
}
(OUT_DIR / 'official_sources_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)
print('Save this notebook output as Kaggle Dataset: freshguard-official-sources-v2')
